# LPC-SonoNet calibrated benchmark on the locked QGECAF patient folds

This runner trains ten TensorFlow LPC-SonoNet models, fits fold-specific temperature scaling using only inner-validation logits, and compares the calibrated results with the completed QGECAF experiment. The untouched final-test cohort is not evaluated.

Place this notebook and `LPC_SonoNet_Calibrated_10Fold_Automated.py` in the same folder. Select the `chaitalienv` kernel, restart it before this experiment, and run the cells in order. This is a documented TensorFlow reimplementation under the locked patient-fold protocol, not a reproduction of the source paper's random split.

In [1]:
import os
os.environ['TF_DETERMINISTIC_OPS'] = '0'

from pathlib import Path
import pandas as pd
from LPC_SonoNet_Calibrated_10Fold_Automated import Config, run
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.10.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
cfg = Config(
    qgecaf_root=(
        r'D:\chaitali\qgacef'
        r'\QGECAF_REVIEWER_REVISION_NESTED_BGWO'
    ),
    output=(
        r'D:\chaitali\qgacef'
        r'\LPC_SONONET_CALIBRATED_SAME_PATIENT_FOLDS'
    ),
    image_size=128,
    num_classes=6,
    outer_folds=10,
    seed=42,
    batch_size=8,
    epochs=50,
    learning_rate=1e-4,
    lpc_channels=(64, 128, 256, 512),
    lpc_branch_ratio=0.5,
    dropout=0.30,
    early_stopping_patience=8,
    resume=True,
)
cfg

Config(qgecaf_root='D:\\chaitali\\qgacef\\QGECAF_REVIEWER_REVISION_NESTED_BGWO', output='D:\\chaitali\\qgacef\\LPC_SONONET_CALIBRATED_SAME_PATIENT_FOLDS', image_size=128, num_classes=6, outer_folds=10, seed=42, batch_size=8, epochs=50, learning_rate=0.0001, dropout=0.3, early_stopping_patience=8, temperature_min=0.05, temperature_max=100.0, ece_bins=15, bootstrap_repetitions=10000, resume=True, smoke_test=False, lpc_channels=(64, 128, 256, 512), lpc_branch_ratio=0.5)

In [3]:
reference_root = Path(cfg.qgecaf_root)
required = [
    reference_root / 'locked_patient_fold_assignments.csv',
    reference_root / 'fold_metrics.csv',
    reference_root / 'development_out_of_fold_predictions.csv',
]
for path in required:
    print(path, '->', path.is_file())
    assert path.is_file(), f'Missing required file: {path}'

for fold in range(1, 11):
    audit = (reference_root / 'cross_validation' / f'fold_{fold:02d}'
             / 'patient_partition_audit.csv')
    assert audit.is_file(), f'Missing fold audit: {audit}'

print('PASS: Reference metrics, locked manifest and all ten partition audits exist.')

D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\locked_patient_fold_assignments.csv -> True
D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\fold_metrics.csv -> True
D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\development_out_of_fold_predictions.csv -> True
PASS: Reference metrics, locked manifest and all ten partition audits exist.


## Run or safely resume

This is the only long-running cell. Re-running it with `resume=True` verifies completed folds and skips them. Do not change the output path or protocol settings during a run.

In [4]:
run(cfg)

TensorFlow 2.10.0; GPUs detected: 1
Reproducibility: fixed seeds and deterministic input ordering; strict GPU-op determinism disabled for TensorFlow 2.10 compatibility.

LPC-SONONET OUTER FOLD 1/10
Instructions for updating:
Use `tf.compat.v1.graph_util.tensor_shape_from_node_def_name`
Epoch 1/50
1143/1143 [==============================] - ETA: 0s - loss: 0.5155 - accuracy: 0.7170
Epoch 1: val_loss improved from inf to 0.86453, saving model to D:\chaitali\qgacef\LPC_SONONET_CALIBRATED_SAME_PATIENT_FOLDS\cross_validation\fold_01\best_model.keras
1143/1143 [==============================] - 99s 70ms/step - loss: 0.5155 - accuracy: 0.7170 - val_loss: 0.8645 - val_accuracy: 0.7203 - lr: 1.0000e-04
Epoch 2/50
1142/1143 [============================>.] - ETA: 0s - loss: 0.3132 - accuracy: 0.8332
Epoch 2: val_loss improved from 0.86453 to 0.40689, saving model to D:\chaitali\qgacef\LPC_SONONET_CALIBRATED_SAME_PATIENT_FOLDS\cross_validation\fold_01\best_model.keras
1143/1143 [================

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



1129/1130 [============================>.] - ETA: 0s - loss: 0.0211 - accuracy: 0.9901
Epoch 16: val_loss did not improve from 0.24010
1130/1130 [==============================] - 61s 54ms/step - loss: 0.0211 - accuracy: 0.9901 - val_loss: 0.2548 - val_accuracy: 0.9323 - lr: 2.5000e-05
Epoch 17/50
1129/1130 [============================>.] - ETA: 0s - loss: 0.0183 - accuracy: 0.9922
Epoch 17: val_loss did not improve from 0.24010
1130/1130 [==============================] - 61s 54ms/step - loss: 0.0183 - accuracy: 0.9923 - val_loss: 0.3196 - val_accuracy: 0.9188 - lr: 2.5000e-05
Epoch 18/50
1129/1130 [============================>.] - ETA: 0s - loss: 0.0154 - accuracy: 0.9940
Epoch 18: val_loss did not improve from 0.24010

Epoch 18: ReduceLROnPlateau reducing learning rate to 1.249999968422344e-05.
1130/1130 [==============================] - 61s 54ms/step - loss: 0.0153 - accuracy: 0.9940 - val_loss: 0.2766 - val_accuracy: 0.9224 - lr: 2.5000e-05
Epoch 19/50
1130/1130 [==============

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



1131/1132 [============================>.] - ETA: 0s - loss: 0.1692 - accuracy: 0.9019
Epoch 4: val_loss improved from 0.59568 to 0.44479, saving model to D:\chaitali\qgacef\LPC_SONONET_CALIBRATED_SAME_PATIENT_FOLDS\cross_validation\fold_05\best_model.keras
1132/1132 [==============================] - 62s 55ms/step - loss: 0.1693 - accuracy: 0.9018 - val_loss: 0.4448 - val_accuracy: 0.8606 - lr: 1.0000e-04
Epoch 5/50
1131/1132 [============================>.] - ETA: 0s - loss: 0.1531 - accuracy: 0.9111
Epoch 5: val_loss did not improve from 0.44479
1132/1132 [==============================] - 62s 55ms/step - loss: 0.1535 - accuracy: 0.9110 - val_loss: 0.6891 - val_accuracy: 0.8039 - lr: 1.0000e-04
Epoch 6/50
1131/1132 [============================>.] - ETA: 0s - loss: 0.1277 - accuracy: 0.9222
Epoch 6: val_loss did not improve from 0.44479
1132/1132 [==============================] - 63s 56ms/step - loss: 0.1276 - accuracy: 0.9222 - val_loss: 1.2983 - val_accuracy: 0.6506 - lr: 1.0000e

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



1135/1135 [==============================] - ETA: 0s - loss: 0.5402 - accuracy: 0.6955
Epoch 1: val_loss improved from inf to 0.95681, saving model to D:\chaitali\qgacef\LPC_SONONET_CALIBRATED_SAME_PATIENT_FOLDS\cross_validation\fold_06\best_model.keras
1135/1135 [==============================] - 66s 56ms/step - loss: 0.5402 - accuracy: 0.6955 - val_loss: 0.9568 - val_accuracy: 0.6741 - lr: 1.0000e-04
Epoch 2/50
1134/1135 [============================>.] - ETA: 0s - loss: 0.3286 - accuracy: 0.8247
Epoch 2: val_loss did not improve from 0.95681
1135/1135 [==============================] - 63s 55ms/step - loss: 0.3286 - accuracy: 0.8247 - val_loss: 1.3799 - val_accuracy: 0.6670 - lr: 1.0000e-04
Epoch 3/50
1134/1135 [============================>.] - ETA: 0s - loss: 0.2449 - accuracy: 0.8668
Epoch 3: val_loss improved from 0.95681 to 0.56379, saving model to D:\chaitali\qgacef\LPC_SONONET_CALIBRATED_SAME_PATIENT_FOLDS\cross_validation\fold_06\best_model.keras
1135/1135 [=================

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [5]:
result_root = Path(cfg.output)
display(pd.read_csv(result_root / 'summary_uncalibrated_mean_sd_ci95.csv'))
display(pd.read_csv(result_root / 'summary_temperature_scaled_mean_sd_ci95.csv'))
display(pd.read_csv(result_root / 'paired_qgecaf_vs_calibrated_lpc_sononet.csv'))
print('Run complete:', (result_root / 'RUN_COMPLETE.json').is_file())

,metric,folds,mean,standard_deviation,ci95_low,ci95_high
0,accuracy,10,0.941822,0.012323,0.933007,0.950637
1,balanced_accuracy,10,0.943410,0.017310,0.931028,0.955793
2,macro_precision,10,0.927896,0.012354,0.919059,0.936733
3,macro_recall,10,0.943410,0.017310,0.931028,0.955793
4,macro_f1,10,0.934395,0.012339,0.925568,0.943222
5,weighted_f1,10,0.941957,0.012464,0.933040,0.950873
6,cohen_kappa,10,0.925092,0.015289,0.914155,0.936029
7,mcc,10,0.925475,0.015263,0.914557,0.936393
8,macro_specificity,10,0.987669,0.002732,0.985715,0.989623
9,ece_15_bins,10,0.024351,0.007303,0.019127,0.029575


,metric,folds,mean,standard_deviation,ci95_low,ci95_high
0,accuracy,10,0.941822,0.012323,0.933007,0.950637
1,balanced_accuracy,10,0.943410,0.017310,0.931028,0.955793
2,macro_precision,10,0.927896,0.012354,0.919059,0.936733
3,macro_recall,10,0.943410,0.017310,0.931028,0.955793
4,macro_f1,10,0.934395,0.012339,0.925568,0.943222
5,weighted_f1,10,0.941957,0.012464,0.933040,0.950873
6,cohen_kappa,10,0.925092,0.015289,0.914155,0.936029
7,mcc,10,0.925475,0.015263,0.914557,0.936393
8,macro_specificity,10,0.987669,0.002732,0.985715,0.989623
9,ece_15_bins,10,0.018886,0.006193,0.014455,0.023316


,metric,qgecaf_mean,calibrated_lpc_sononet_mean,mean_difference_qgecaf_minus_baseline,paired_bootstrap_ci95_low,paired_bootstrap_ci95_high,wilcoxon_statistic,raw_p,rank_biserial_qgecaf_minus_baseline,preferred_direction,holm_adjusted_p,significant_at_0.05
0,macro_f1,0.911895,0.934395,-0.022499,-0.032749,-0.014153,0.0,0.001953,-1.000000,higher,0.007812,True
1,negative_log_likelihood,0.298073,0.193578,0.104495,0.084152,0.125629,0.0,0.001953,1.000000,lower,0.007812,True
2,brier_multiclass,0.121129,0.093128,0.028001,0.018005,0.038942,0.0,0.001953,1.000000,lower,0.007812,True
3,ece_15_bins,0.029862,0.018886,0.010977,0.002982,0.018644,8.0,0.048828,0.709091,lower,0.048828,True


Run complete: True
